# Team Zypher — Modeling & Inference
**Prerequisite:** Run Bronze → Silver → Gold first. This notebook reads `master_feature_store.parquet`, trains LightGBM with a temporal split, builds the **January 2026** inference frame, and writes **`TeamZypher_predictions.csv`**.

| Step | Split |
|------|--------|
| Train | 2023-01 → 2025-10 |
| Validate | 2025-11 → 2025-12 |
| Predict | 2026-01 (all outlets) |

In [4]:
from pathlib import Path

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive", force_remount=True)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q lightgbm scikit-learn pyarrow

In [5]:
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - [%(levelname)s] - %(message)s",
    force=True,
)

# --- Configuration ---
TARGET_COL = "Latent_Demand_Proxy"
DATE_COL = "Date"
GROUP_COLS = ["Outlet_ID", "SKU_ID"]

INFERENCE_YEAR = 2026
INFERENCE_MONTH = 1
INFERENCE_DATE = pd.Timestamp(f"{INFERENCE_YEAR}-{INFERENCE_MONTH:02d}-01")

TRAIN_END = pd.Timestamp("2025-10-31")
VAL_START = pd.Timestamp("2025-11-01")
VAL_END = pd.Timestamp("2025-12-31")

STATIC_FEATURE_COLS = [
    "Outlet_Size",
    "Cooler_Count",
    "Outlet_Type",
    "Lat",
    "Lon",
    "POI_Count_Within_2km",
    "Distributor_ID",
]

MODEL_FEATURE_COLS = [
    "Holiday_Count",
    "Seasonality_Index",
    "Lag_1M_Volume",
    "Lag_2M_Volume",
    "Rolling_6M_Max_Volume",
    "Outlet_Size",
    "Cooler_Count",
    "Outlet_Type",
    "Lat",
    "Lon",
    "POI_Count_Within_2km",
    "Distributor_ID",
    "SKU_ID",
    "Month",
]

CATEGORICAL_FEATURES = [
    "Outlet_Size",
    "Outlet_Type",
    "Distributor_ID",
    "SKU_ID",
]

OUTPUT_CSV_NAME = "TeamZypher_predictions.csv"

In [6]:
@dataclass(frozen=True)
class PipelinePaths:
    base_dir: Path
    feature_store: Path
    holiday_csv: Path | None
    output_csv: Path


def find_latest_parquet(gold_dir: Path, filename: str) -> Path:
    if not gold_dir.exists():
        raise FileNotFoundError(f"Gold directory not found: {gold_dir}")

    runs = sorted(
        (d for d in gold_dir.iterdir() if d.is_dir() and d.name[:2].isdigit()),
        key=lambda p: p.name,
        reverse=True,
    )
    for run_dir in runs:
        candidate = run_dir / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {filename} under any timestamped run in {gold_dir}"
    )


def find_holiday_csv(base_dir: Path) -> Path | None:
    bronze = base_dir / "lakehouse" / "bronze"
    if not bronze.exists():
        return None

    direct = bronze / "holiday_list.csv"
    if direct.exists():
        return direct

    runs = sorted(
        (d for d in bronze.iterdir() if d.is_dir()),
        key=lambda p: p.name,
        reverse=True,
    )
    for run_dir in runs:
        candidate = run_dir / "holiday_list.csv"
        if candidate.exists():
            return candidate
    return None


def build_paths(
    base_dir: Path,
    feature_store: str | None = None,
    output_csv: str | None = None,
) -> PipelinePaths:
    gold_dir = base_dir / "lakehouse" / "gold"
    parquet_path = (
        Path(feature_store).expanduser().resolve()
        if feature_store
        else find_latest_parquet(gold_dir, "master_feature_store.parquet")
    )
    out_path = (
        Path(output_csv).expanduser().resolve()
        if output_csv
        else base_dir / OUTPUT_CSV_NAME
    )
    return PipelinePaths(
        base_dir=base_dir,
        feature_store=parquet_path,
        holiday_csv=find_holiday_csv(base_dir),
        output_csv=out_path,
    )

In [7]:
def load_feature_store(path: Path) -> pd.DataFrame:
    logging.info("Loading feature store: %s", path)
    df = pd.read_parquet(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df["Year"] = df[DATE_COL].dt.year.astype("int16")
    df["Month"] = df[DATE_COL].dt.month.astype("int8")
    return df


def approximate_imputed_volume(df: pd.DataFrame) -> pd.Series:
    seasonality = pd.to_numeric(df["Seasonality_Index"], errors="coerce").replace(0, np.nan)
    seasonality = seasonality.fillna(1.0).astype("float32")
    latent = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
    return (latent / seasonality).clip(lower=0.0).astype("float32")


def attach_imputed_volume(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_Volume_Imputed_Proxy"] = approximate_imputed_volume(out)
    return out


def temporal_mask(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    return (df[DATE_COL] >= start) & (df[DATE_COL] <= end)


def compute_rolling_6m_max(
    history: pd.DataFrame,
    as_of: pd.Timestamp,
    volume_col: str = "_Volume_Imputed_Proxy",
) -> pd.DataFrame:
    subset = history[history[DATE_COL] <= as_of].copy()
    subset = subset.sort_values(GROUP_COLS + [DATE_COL])

    def _window_max(series: pd.Series) -> float:
        return float(series.tail(6).max())

    return (
        subset.groupby(GROUP_COLS, observed=True)[volume_col]
        .apply(_window_max)
        .reset_index(name="Rolling_6M_Max_Volume")
    )


def month_holiday_count_from_store(df: pd.DataFrame, year: int, month: int) -> int:
    subset = df[(df["Year"] == year) & (df["Month"] == month)]
    if subset.empty:
        raise ValueError(f"No Holiday_Count found for {year}-{month:02d} in feature store.")
    return int(subset["Holiday_Count"].iloc[0])


def month_holiday_count_from_csv(holiday_csv: Path, year: int, month: int) -> int:
    holidays = pd.read_csv(holiday_csv)
    holidays["Date"] = pd.to_datetime(holidays["Date"])
    mask = (holidays["Date"].dt.year == year) & (holidays["Date"].dt.month == month)
    return int(mask.sum())


def resolve_jan_holiday_count(df: pd.DataFrame, holiday_csv: Path | None) -> int:
    try:
        return month_holiday_count_from_store(df, INFERENCE_YEAR, INFERENCE_MONTH)
    except ValueError:
        pass

    if holiday_csv is not None:
        try:
            return month_holiday_count_from_csv(holiday_csv, INFERENCE_YEAR, INFERENCE_MONTH)
        except Exception as exc:
            logging.warning("Holiday CSV lookup failed (%s); using Jan 2025 fallback.", exc)

    logging.warning(
        "Jan %s holiday count unavailable; falling back to Jan 2025 from feature store.",
        INFERENCE_YEAR,
    )
    return month_holiday_count_from_store(df, 2025, INFERENCE_MONTH)


def jan_seasonality_by_distributor(df: pd.DataFrame) -> pd.DataFrame:
    jan_rows = df[df["Month"] == INFERENCE_MONTH].copy()
    if jan_rows.empty:
        raise ValueError("No January rows found to derive seasonality.")

    jan_rows = jan_rows.sort_values(["Distributor_ID", "Year"])
    seasonality = (
        jan_rows.groupby("Distributor_ID", observed=True)["Seasonality_Index"]
        .last()
        .reset_index()
    )
    seasonality["Seasonality_Index"] = pd.to_numeric(
        seasonality["Seasonality_Index"], errors="coerce"
    ).fillna(1.0)
    return seasonality

In [8]:
def build_jan_2026_inference_frame(
    df: pd.DataFrame,
    holiday_csv: Path | None,
) -> pd.DataFrame:
    """Construct SKU-outlet rows for January 2026 inference."""
    history = attach_imputed_volume(df)
    history = history.sort_values(GROUP_COLS + [DATE_COL])

    dec_date = pd.Timestamp("2025-12-01")
    nov_date = pd.Timestamp("2025-11-01")

    dec_rows = history[history[DATE_COL] == dec_date][GROUP_COLS + ["_Volume_Imputed_Proxy"]]
    nov_rows = history[history[DATE_COL] == nov_date][GROUP_COLS + ["_Volume_Imputed_Proxy"]]

    if dec_rows.empty or nov_rows.empty:
        raise ValueError("History must include Nov and Dec 2025 for lag construction.")

    dec_rows = dec_rows.rename(columns={"_Volume_Imputed_Proxy": "Lag_1M_Volume"})
    nov_rows = nov_rows.rename(columns={"_Volume_Imputed_Proxy": "Lag_2M_Volume"})

    latest_idx = history.groupby(GROUP_COLS, observed=True)[DATE_COL].idxmax()
    latest = history.loc[latest_idx].copy()

    future = latest[GROUP_COLS + STATIC_FEATURE_COLS].copy()
    future[DATE_COL] = INFERENCE_DATE
    future["Year"] = INFERENCE_YEAR
    future["Month"] = INFERENCE_MONTH

    future = future.merge(dec_rows, on=GROUP_COLS, how="left")
    future = future.merge(nov_rows, on=GROUP_COLS, how="left")
    future["Lag_1M_Volume"] = future["Lag_1M_Volume"].fillna(0.0).astype("float32")
    future["Lag_2M_Volume"] = future["Lag_2M_Volume"].fillna(0.0).astype("float32")

    future["Holiday_Count"] = np.int16(resolve_jan_holiday_count(df, holiday_csv))

    seasonality = jan_seasonality_by_distributor(df)
    future = future.merge(seasonality, on="Distributor_ID", how="left")
    future["Seasonality_Index"] = future["Seasonality_Index"].fillna(1.0).astype("float32")

    rolling = compute_rolling_6m_max(history, as_of=dec_date)
    future = future.merge(rolling, on=GROUP_COLS, how="left")
    future["Rolling_6M_Max_Volume"] = future["Rolling_6M_Max_Volume"].fillna(0.0).astype("float32")

    return future

In [9]:
def prepare_training_matrix(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    categorical_cols: Sequence[str],
) -> tuple[pd.DataFrame, list[str]]:
    matrix = df[list(feature_cols)].copy()
    cat_cols = [c for c in categorical_cols if c in matrix.columns]
    for col in cat_cols:
        matrix[col] = matrix[col].astype("category")
    return matrix, cat_cols


def train_lightgbm(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_val: pd.DataFrame,
    y_val: pd.Series,
    categorical_features: Iterable[str],
) -> lgb.LGBMRegressor:
    model = lgb.LGBMRegressor(
        objective="regression",
        boosting_type="gbdt",
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=40,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    callbacks = [
        lgb.early_stopping(stopping_rounds=75, verbose=False),
        lgb.log_evaluation(period=100),
    ]

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_val, y_val)],
        eval_metric="rmse",
        categorical_feature=list(categorical_features),
        callbacks=callbacks,
    )
    return model


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    return {"rmse": rmse, "mae": mae}


def aggregate_outlet_predictions(
    sku_predictions: pd.DataFrame,
    id_col: str = "Outlet_ID",
    pred_col: str = "predicted_latent",
) -> pd.DataFrame:
    outlet = (
        sku_predictions.groupby(id_col, observed=True)[pred_col]
        .sum()
        .reset_index()
        .rename(columns={pred_col: "Maximum_Monthly_Liters"})
    )
    outlet["Maximum_Monthly_Liters"] = outlet["Maximum_Monthly_Liters"].clip(lower=0.0)
    return outlet


def run_pipeline(paths: PipelinePaths) -> pd.DataFrame:
    df = load_feature_store(paths.feature_store)
    logging.info("Feature store shape: %s", df.shape)

    train_df = df[temporal_mask(df, df[DATE_COL].min(), TRAIN_END)].copy()
    val_df = df[temporal_mask(df, VAL_START, VAL_END)].copy()
    logging.info("Train rows: %s | Validation rows: %s", len(train_df), len(val_df))

    train_x, cat_cols = prepare_training_matrix(train_df, MODEL_FEATURE_COLS, CATEGORICAL_FEATURES)
    val_x, _ = prepare_training_matrix(val_df, MODEL_FEATURE_COLS, CATEGORICAL_FEATURES)

    train_y = train_df[TARGET_COL].astype("float32")
    val_y = val_df[TARGET_COL].astype("float32")

    model = train_lightgbm(train_x, train_y, val_x, val_y, cat_cols)
    logging.info("Best iteration: %s", model.best_iteration_)

    val_pred = model.predict(val_x, num_iteration=model.best_iteration_)
    metrics = evaluate_predictions(val_y.to_numpy(), val_pred)
    logging.info("Validation RMSE: %.4f | MAE: %.4f", metrics["rmse"], metrics["mae"])

    full_train = pd.concat([train_df, val_df], ignore_index=True)
    full_x, _ = prepare_training_matrix(full_train, MODEL_FEATURE_COLS, CATEGORICAL_FEATURES)
    full_y = full_train[TARGET_COL].astype("float32")

    final_model = lgb.LGBMRegressor(**model.get_params())
    final_model.set_params(n_estimators=model.best_iteration_ or model.n_estimators)
    final_model.fit(full_x, full_y, categorical_feature=cat_cols)

    inference_df = build_jan_2026_inference_frame(df, paths.holiday_csv)
    logging.info("Jan 2026 inference frame shape: %s", inference_df.shape)

    infer_x, _ = prepare_training_matrix(inference_df, MODEL_FEATURE_COLS, CATEGORICAL_FEATURES)
    sku_pred = inference_df[GROUP_COLS].copy()
    sku_pred["predicted_latent"] = final_model.predict(infer_x)
    sku_pred["predicted_latent"] = sku_pred["predicted_latent"].clip(lower=0.0)

    submission = aggregate_outlet_predictions(sku_pred)
    submission = submission[["Outlet_ID", "Maximum_Monthly_Liters"]]

    if submission.isnull().any().any():
        raise ValueError("Submission contains null values.")
    if (submission["Maximum_Monthly_Liters"] < 0).any():
        raise ValueError("Submission contains negative predictions.")

    submission.to_csv(paths.output_csv, index=False)
    logging.info("Wrote submission file: %s (%s outlets)", paths.output_csv, len(submission))
    return submission

In [ ]:
BASE_DIR = "/content/drive/MyDrive/DataStorm_TeamZypher"

FEATURE_STORE_PATH = None
OUTPUT_CSV_PATH = None

paths = build_paths(Path(BASE_DIR), FEATURE_STORE_PATH, OUTPUT_CSV_PATH)

logging.info("Using base directory: %s", paths.base_dir)
logging.info("Feature store: %s", paths.feature_store)
if paths.holiday_csv:
    logging.info("Holiday reference: %s", paths.holiday_csv)

submission = run_pipeline(paths)
submission.head(10)

2026-05-16 23:46:47,619 - [INFO] - Using base directory: /content/drive/MyDrive/DataStorm_TeamZypher
2026-05-16 23:46:47,622 - [INFO] - Feature store: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/gold/20260516_233646/master_feature_store.parquet
2026-05-16 23:46:47,623 - [INFO] - Holiday reference: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/bronze/20260516_225859/holiday_list.csv
2026-05-16 23:46:47,624 - [INFO] - Loading feature store: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/gold/20260516_233646/master_feature_store.parquet
2026-05-16 23:47:00,580 - [INFO] - Feature store shape: (7182720, 23)
2026-05-16 23:47:03,496 - [INFO] - Train rows: 6783680 | Validation rows: 399040


[100]	valid_0's rmse: 17.1517	valid_0's l2: 294.181
[200]	valid_0's rmse: 16.8399	valid_0's l2: 283.583


In [ ]:
print(f"Rows: {len(submission):,}")
print(f"Columns: {list(submission.columns)}")
print(f"Min prediction: {submission['Maximum_Monthly_Liters'].min():.4f}")
print(f"Max prediction: {submission['Maximum_Monthly_Liters'].max():.4f}")
print(f"Saved to: {paths.output_csv}")